# MS gate: radial rocking / rocking + radial COM

In [ ]:
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
from IPython.display import display

import rocking_com_model as rc
rc = importlib.reload(rc)

In [ ]:
USE_TWO_MODES = True       # True: rocking + radial COM / False: rocking only
EXECUTE_QPT = True        # QPT を実行：True
USE_FULL_ORDER = True      # LD paraの2次まで：True / 1次まで：False
SAVE_RESULTS = True

In [ ]:
MODE_PARAMETERS = {
    "axial_frequency_hz": rc.PAPER_PARAMETERS["axial_frequency_hz"],
    "com_frequency_hz": rc.PAPER_PARAMETERS["radial_high_com_frequency_hz"],
    "rocking_frequency_hz": None,      # sqrt(f_COM**2 - f_axial**2)
    "radial_projection": 1.0,         # 仮定: 差波数は選んだ動径軸に平行
    "eta_com": None,                  # eta_rocking * sqrt(f_rocking / f_COM)
    "n_bar_com": 0.01,                # 仮定。論文の輸送励起量0.01とは別の初期平均占有数
    "com_heating_rate_phys": 10.0,    # quanta/s。論文の範囲内で置く仮定
    "com_dephasing_rate_phys": 0.0,   # 未同定のためこの効果を無効化
    "com_phonon_dim_override": 8,
}

In [ ]:
PULSE_PARAMETERS = rc.two_loop_ms_pulse(
    detuning_hz=rc.PAPER_PARAMETERS["reference_detuning_magnitude_hz"],
    gate_duration_s=rc.PAPER_PARAMETERS["reference_gate_duration_s"],
    delta_sim=0.5,
)

BASE_PARAMETERS = {
    **PULSE_PARAMETERS,
    "rho0": 0.0,
    "n_bar_list": [0.01],             # rocking の初期平均フォノン数。論文値ではなく従来の仮定
    "time_points": 500,              # 制御波形用格子。ON/OFF境界はこれと独立に厳密に区切る
    "eta": 0.1,
    "heating_rate_phys": 10.0,
    "dephasing_rate_phys": 18.0,
    "T2_star": 0.3,
    "rayleigh_rate_phys": 3.0,       
    "raman_rate_phys": 1.0, 
    "laser_intensity_fluctuation": 0.0,
    "laser_detuning_fluctuation": 0.0,  # simulation angular-frequency units
    "laser_rotation_angle_fluctuation": 0.0,
    "laser_noise_samples": 1,
    "laser_noise_seed": 1234,
    "use_full_order": USE_FULL_ORDER,
    "laser_scattering_scales_with_intensity": False,
    "scattering_reference_amplitude": None,
    "phonon_dim_override": 16,       # rocking cutoff; spectator COM is 8 above
    "solver_max_step": None,
    "parallel_workers": 1,
    "show_progress": True,
}
IDEAL_PHI = -np.pi / 4

In [ ]:
PLAN = rc.prepare_simulation(
    use_two_modes=USE_TWO_MODES,
    **BASE_PARAMETERS,
    **MODE_PARAMETERS,
)

print("Model:", "rocking + radial COM (spectator)" if USE_TWO_MODES else "rocking only")
CONDITIONS = pd.DataFrame(PLAN["conditions"])
CONDITIONS["initial_rocking_remain"] = 1.0 - CONDITIONS["initial_rocking_tail"]
if USE_TWO_MODES:
    CONDITIONS["initial_com_remain"] = 1.0 - CONDITIONS["initial_com_tail"]
display(CONDITIONS)
display(pd.Series({
    k: v for k, v in PLAN["modes"].items()
    if k not in ("delta_com_sim", "drive_frequency_hz")
}, name="Mode parameters"))

SCALE = PLAN["rates"]["scale_factor"]
display(pd.DataFrame([{
    "start_us": segment["start"] * SCALE * 1e6,
    "stop_us": segment["stop"] * SCALE * 1e6,
    "laser_on": segment["drive_on"],
} for segment in PLAN["segments"]]))
print("Rocking detuning (Hz):", PLAN["detuning"][[0, -1]] / (2 * np.pi * SCALE))
if USE_TWO_MODES:
    print("COM detuning (Hz):", PLAN["modes"]["delta_com_sim"][[0, -1]] / (2 * np.pi * SCALE))
print("Drive frequency (Hz):", PLAN["modes"]["drive_frequency_hz"][[0, -1]])
print("Solver max_step (physical seconds):", PLAN["solver_max_step"] * SCALE)

In [ ]:
RESULT = rc.run_analysis(
    execute=EXECUTE_QPT,
    use_two_modes=USE_TWO_MODES,
    ideal_phi=IDEAL_PHI,
    show_plot=False,
    error_channel_convention="undo_before_actual",
    **BASE_PARAMETERS,
    **MODE_PARAMETERS,
)
if RESULT is None:
    print("EXECUTE_QPT=False: 時間発展・QPT は実行していません。")

In [ ]:
if RESULT is not None:
    display(rc.summary_table(RESULT))
    FIGURES = rc.plot_results(RESULT, index=0)
else:
    FIGURES = None

In [ ]:
MODEL_TAG = "rocking_com" if USE_TWO_MODES else "rocking_only"
OUTPUT_DIR = Path("results") / "com_stretch_ms_gate" / MODEL_TAG / "run_001"

SAVED_DIRECTORY = None
if SAVE_RESULTS and RESULT is not None:
    SAVED_DIRECTORY = rc.save_results(RESULT, OUTPUT_DIR)
    print("Saved:", SAVED_DIRECTORY)